# 011 — RAPTOR: Hierarchical Summarization Retrieval
**Retrieval Series** | Multi-level tree of summaries for broad questions

**RAPTOR** (Recursive Abstractive Processing for Tree-Organized Retrieval):
- Level 0: original leaf chunks
- Level 1: LLM summaries of clusters of similar chunks
- Level 2: summaries of level-1 summaries
- Retrieval queries ALL levels → answers at any granularity


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Level 0: leaf chunks ─────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import numpy as np

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)
leaf_chunks = splitter.split_text(ai_text[:12000])
print(f"Level 0 (leaves): {len(leaf_chunks)} chunks")


Level 0 (leaves): 15 chunks


In [5]:
# ── Cluster helper: group by embedding similarity ────────────────────────
def cluster_texts(texts, n_clusters=None):
    vecs = np.array(embeddings.embed_documents(texts))
    n = n_clusters or max(2, len(texts) // 5)

    # Simple greedy clustering by cosine similarity
    assigned = [-1] * len(texts)
    cluster_centers = []
    cluster_id = 0

    for i, vec in enumerate(vecs):
        if assigned[i] != -1:
            continue
        cluster_centers.append((cluster_id, vec))
        assigned[i] = cluster_id
        for j in range(i + 1, len(texts)):
            if assigned[j] != -1:
                continue
            sim = float(np.dot(vec, vecs[j]) /
                        (np.linalg.norm(vec) * np.linalg.norm(vecs[j]) + 1e-10))
            if sim > 0.75:
                assigned[j] = cluster_id
        cluster_id += 1
        if cluster_id >= n:
            for k in range(len(texts)):
                if assigned[k] == -1:
                    assigned[k] = cluster_id - 1
            break

    clusters: dict[int, list] = {}
    for i, cid in enumerate(assigned):
        clusters.setdefault(cid, []).append(texts[i])
    return clusters

clusters = cluster_texts(leaf_chunks, n_clusters=6)
print(f"Clusters formed: {len(clusters)}")
for cid, members in clusters.items():
    print(f"  Cluster {cid}: {len(members)} chunks")


Clusters formed: 6
  Cluster 0: 2 chunks
  Cluster 1: 1 chunks
  Cluster 2: 1 chunks
  Cluster 3: 1 chunks
  Cluster 4: 1 chunks
  Cluster 5: 9 chunks


In [6]:
# ── Level 1: summarize each cluster ──────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summarize_prompt = ChatPromptTemplate.from_template(
    "Write a concise 3-4 sentence summary of the following passages:\n\n{text}\n\nSummary:"
)
summarize_chain = summarize_prompt | llm | StrOutputParser()

level1_summaries = []
for cid, members in clusters.items():
    combined = "\n\n".join(members)[:3000]
    summary  = summarize_chain.invoke({"text": combined})
    level1_summaries.append(summary)
    print(f"  Cluster {cid} summary: {summary[:100]}...")

print(f"\nLevel 1 summaries: {len(level1_summaries)}")


  Cluster 0 summary: Here's a concise 3-4 sentence summary of the passages:

Artificial intelligence (AI) refers to the a...


  Cluster 1 summary: Here's a concise 3-4 sentence summary:

Artificial Intelligence (AI) has become increasingly prevale...


  Cluster 2 summary: The field of Artificial Intelligence (AI) research was founded at a 1956 workshop at Dartmouth Colle...


  Cluster 3 summary: Here's a concise 3-4 sentence summary:

The first AI winter occurred from 1974 to 1980, marked by a ...


  Cluster 4 summary: In the 1980s, AI research experienced a revival with the emergence of expert systems. These systems ...


  Cluster 5 summary: Here's a concise 3-4 sentence summary of the passages:

The field of artificial intelligence (AI) wa...

Level 1 summaries: 6


In [7]:
# ── Level 2: meta-summary ────────────────────────────────────────────────
combined_l1 = "\n\n".join(level1_summaries)
level2_summary = summarize_chain.invoke({"text": combined_l1[:3000]})
print(f"Level 2 (document meta-summary):\n{level2_summary}")


Level 2 (document meta-summary):
Here's a concise 4-sentence summary of the passages:

Artificial intelligence (AI) refers to the ability of machines to demonstrate intelligence, with research focusing on creating intelligent agents that perceive their environment and achieve goals. AI has made significant advancements in various applications, including search engines, self-driving cars, and voice assistants, and has transformed the way we interact with technology. The field of AI research has experienced periods of growth and decline, including the first AI winter from 1974 to 1980, but has seen a revival with the emergence of expert systems and deep learning in the 1980s and 2010s. The development of AI has been marked by significant milestones, including the success of deep learning and the use of artificial neural networks for tasks such as image recognition and pattern recognition.


In [8]:
# ── Build multi-level RAPTOR index ───────────────────────────────────────
from langchain_community.vectorstores import FAISS

# Combine all levels for retrieval
all_nodes = (
    [Document(page_content=c, metadata={"level": 0}) for c in leaf_chunks] +
    [Document(page_content=s, metadata={"level": 1}) for s in level1_summaries] +
    [Document(page_content=level2_summary, metadata={"level": 2})]
)

raptor_store = FAISS.from_documents(all_nodes, embeddings)
raptor_ret   = raptor_store.as_retriever(search_kwargs={"k": 6})
print(f"✓ RAPTOR index: {len(all_nodes)} nodes across 3 levels")


✓ RAPTOR index: 22 nodes across 3 levels


In [9]:
# ── Query the RAPTOR tree ────────────────────────────────────────────────
def raptor_query(question: str):
    results = raptor_ret.invoke(question)
    by_level: dict[int, list] = {}
    for r in results:
        lvl = r.metadata["level"]
        by_level.setdefault(lvl, []).append(r.page_content)

    print(f"Results by level for: {question!r}")
    for lvl in sorted(by_level):
        label = ["leaf", "cluster summary", "meta-summary"][lvl]
        print(f"  Level {lvl} ({label}): {len(by_level[lvl])} nodes")
        for text in by_level[lvl][:1]:
            print(f"    → {text[:150]}...")

    ctx = "\n\n".join(r.page_content for r in results[:4])
    ans_chain = (
        ChatPromptTemplate.from_template(
            "Answer the question using the context.\nContext:\n{ctx}\nQ: {q}\nA:"
        ) | llm | StrOutputParser()
    )
    return ans_chain.invoke({"ctx": ctx[:3500], "q": question})

broad_q  = "Provide an overview of the history and key milestones of AI"
narrow_q = "What is the difference between weak and strong AI?"

print("=== Broad question ===")
print(raptor_query(broad_q)[:400])
print()
print("=== Narrow question ===")
print(raptor_query(narrow_q)[:400])


=== Broad question ===
Results by level for: 'Provide an overview of the history and key milestones of AI'
  Level 0 (leaf): 3 nodes
    → AI applications include advanced web search engines such as Google Search, recommendation systems
used by YouTube, Amazon, and Netflix, understanding ...
  Level 1 (cluster summary): 2 nodes
    → Here's a concise 3-4 sentence summary of the passages:

The field of artificial intelligence (AI) was transformed in the 2010s by the success of deep ...
  Level 2 (meta-summary): 1 nodes
    → Here's a concise 4-sentence summary of the passages:

Artificial intelligence (AI) refers to the ability of machines to demonstrate intelligence, with...


The history of Artificial Intelligence (AI) began in 1956 with a workshop at Dartmouth College, where the field's leaders were established. Initially, researchers predicted that a human-like intelligent machine would exist within a generation, but commercial developers and government contractors took a more pragmatic approach. The field experienced periods of growth and decline, including the firs

=== Narrow question ===
Results by level for: 'What is the difference between weak and strong AI?'
  Level 0 (leaf): 3 nodes
    → Artificial Intelligence

Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to natural intelligence
displayed by animal...
  Level 1 (cluster summary): 2 nodes
    → Here's a concise 3-4 sentence summary of the passages:

Artificial intelligence (AI) refers to the ability of machines to demonstrate intelligence, as...
  Level 2 (meta-summary): 1 nodes
    → Here's a concise 4-sentence summary of the passages:

Artificial intelligenc

Unfortunately, the provided context does not explicitly mention the terms "weak AI" and "strong AI." However, based on general knowledge about the topic, I can provide an answer.

Weak AI, also known as narrow AI, refers to a type of artificial intelligence that is designed to perform a specific task, such as image recognition, language translation, or playing chess. Weak AI is trained to excel in


## Key Takeaways
- RAPTOR enables **multi-granularity retrieval**: broad + specific in one index
- Broad questions hit level-1/2 summaries; narrow ones hit leaf chunks
- Summarization cost is paid once at ingest time (worth it for large corpora)
- Original paper: Yu et al., 2024 — shows 20%+ improvement on long-doc QA
